# Hypothesis-Kommentare für eine Redaktionsrunde sammeln

Ziel: kurze Annotationen in eine freizugebende Aktionsliste übersetzen. Verbindliche Regeln und Zielgruppe stehen in `docs/development/website_editorial_workflow.md`.

Voraussetzungen: Windows PowerShell, Internetzugang und der bereits lokal mit DPAPI gespeicherte Token `~/.hypothesis-token.dpapi`. Der Token wird weder angezeigt noch im Notebook gespeichert. Das gemeinsame interne Skript liest die vereinbarte Gruppe, paginiert die Ergebnisse und behält nur URLs der PEExcel-Website. Es verändert keine Annotationen.

Die folgenden Zellen sammeln den aktuellen Stand und zeigen Originalzitat, Kommentar und Link nach Seiten. Danach im Chat je Arbeitspaket die konkrete Aktion, offene Fragen und Bildbedarf formulieren. Erst nach Freigabe die Seiten bearbeiten. Anschließend Vorschau zur Abnahme zeigen; Veröffentlichung und Löschung separat durchführen. Vor einer autorisierten Löschung Annotation-ID und Änderungszeit erneut mit dem freigegebenen Stand vergleichen. Dieses Notebook führt weder Seitenänderungen noch Veröffentlichungen oder Löschungen aus.


In [ ]:
from pathlib import Path
import json
import subprocess
import os
from collections import Counter

root = Path.cwd()
if not (root / "scripts/collect_website_annotations.ps1").exists():
    root = root.parent
# Windows PowerShell must discover its own modules, even inside a PowerShell 7 host.
process_env = os.environ.copy()
for key in list(process_env):
    if key.lower() == "psmodulepath":
        process_env.pop(key)
result = subprocess.run(
    ["powershell.exe", "-NoProfile", "-File", str(root / "scripts/collect_website_annotations.ps1")],
    capture_output=True, encoding="utf-8", check=True, env=process_env,
)
annotations = json.loads(result.stdout.lstrip("\ufeff"))
assert isinstance(annotations, list)
assert len({a["id"] for a in annotations}) == len(annotations)
assert all(a["group"] == "Q4qRzANo" for a in annotations)
assert all(a["uri"].startswith(("https://simonschaluppe.github.io/peexcel/", "http://simonschaluppe.github.io/peexcel/")) for a in annotations)
print(f"{len(annotations)} Annotationen")
print(dict(Counter(a["uri"].rsplit("/", 1)[-1] for a in annotations)))


In [ ]:
for annotation in sorted(annotations, key=lambda a: (a["uri"], a["created"])):
    quotes = [selector["exact"] for target in annotation.get("target", [])
              for selector in target.get("selector", [])
              if selector.get("type") == "TextQuoteSelector"]
    print("\n", annotation["uri"], "\n", annotation["links"]["html"])
    print("Markierung:", " | ".join(quotes) or "Seitennotiz / Antwort")
    print("Kommentar:", annotation["text"])


## Optionale lokale Momentaufnahme

Enthält private Kommentare. Nur in der ignorierten lokalen Ablage behalten; keine Uploads oder öffentlichen Downloads. Das JSON entspricht unverändert den Annotationen der API.


In [ ]:
# Optional: Rohdaten dieser Runde lokal aufbewahren, nicht veröffentlichen.
# Ein neuer Zeitstempel verhindert das Überschreiben früherer Freigabestände.
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
folder = root / "notebooks/local_exports/editorial"
folder.mkdir(parents=True, exist_ok=True)
snapshot = folder / f"hypothesis-{stamp}.json"
snapshot.write_text(json.dumps(annotations, ensure_ascii=False, indent=2), encoding="utf-8")
print(snapshot)
